In [ ]:
import pandas as pd
import re

# Load HZ sheet
hz = pd.read_excel(file_path, sheet_name="HZ")

# Machine columns
machine_cols = [col for col in hz.columns if col.startswith("Machine")]

# Get all unique machines
machines = pd.unique(hz[machine_cols].values.ravel())
machines = [m for m in machines if pd.notna(m)]

# Function to extract tonnage from machine name
def extract_machine_tonnage(machine):
    match = re.search(r'(\d+)T', machine)
    if match:
        return int(match.group(1))
    return None

machine_tonnage = {m: extract_machine_tonnage(m) for m in machines}

# Convert tonnage column into list
def parse_tonnage(x):
    if pd.isna(x):
        return []
    return [int(t.strip()) for t in str(x).split(",")]

hz["Tonnage_List"] = hz["Tonnage"].apply(parse_tonnage)

# Create compatibility matrix
matrix = []

for _, row in hz.iterrows():
    part = row["Part"]
    tonnage_list = row["Tonnage_List"]

    row_data = {"Part": part}

    for machine in machines:
        m_ton = machine_tonnage[machine]

        if m_ton in tonnage_list:
            row_data[machine] = 1
        else:
            row_data[machine] = 0

    matrix.append(row_data)

compatibility_matrix = pd.DataFrame(matrix)

print(compatibility_matrix)